In [ ]:
# Dependencies
%pip install -q xarray cfgrib eccodes matplotlib cartopy requests numpy ecmwf-opendata

import xarray as xr
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import requests
from pathlib import Path
from IPython.display import HTML
from ecmwf.opendata import Client

## Load forecast from ECMWF (IFS)
In this section we'll use forecast output from the European Centre for Medium-Range Weather Forecasts (ECMWF). We do this mainly because the data load is much smaller than what is needed from DMI's IG domain, as the DMI API cannot download data subsets.

In [ ]:
# Download 48h of IFS 00z oper forecast via ecmwf-opendata
client = Client(source="ecmwf", model="ifs", resol="0p25")

# Get the latest available run
latest_run = client.latest(type="fc", stream="oper", step=0)
print(f"Latest IFS run: {latest_run}")

# Retrieve steps 0-48h (3-hourly) into a single GRIB file
ifs_steps = list(range(0, 49, 3))  # [0, 3, 6,... 48]
ifs_target = Path("ifs_data")
ifs_target.mkdir(exist_ok=True)
ifs_grib = ifs_target / "ifs_oper_00z.grib"

if not ifs_grib.exists():
    print(f"Retrieving IFS steps {ifs_steps} (params only) ...")
    client.retrieve(
        type="fc",
        stream="oper",
        step=ifs_steps,
        param=["2t", "tp", "msl", "sf", "10u", "10v"],
        target=str(ifs_grib),
    )
    print(f"  Saved to {ifs_grib}  ({ifs_grib.stat().st_size / 1e6:.0f} MB)")
else:
    print(f"Using cached {ifs_grib}")

# Greenland bounding box definition
LAT_S, LAT_N = 58.0, 85.0
LON_W, LON_E = -75.0, -10.0

def subset_greenland(ds):
    """Subset an xarray dataset to the Greenland domain."""
    lons = ds.longitude.values
    # Handle 0-360 vs -180..180
    if lons.max() > 180:
        ds = ds.assign_coords(longitude=(ds.longitude + 180) % 360 - 180)
        ds = ds.sortby("longitude")
    return ds.sel(
        latitude=slice(LAT_N, LAT_S),
        longitude=slice(LON_W, LON_E),
    )

# Extract 5 parameters and subset to Greenland for example
def load_ifs_param(grib_path, filter_keys, label=""):
    ds = xr.open_dataset(
        grib_path,
        engine="cfgrib",
        backend_kwargs={"filter_by_keys": filter_keys, "indexpath": ""},
    )
    ds = subset_greenland(ds)
    vnames = list(ds.data_vars)
    print(f"  OK {label}: {vnames}, shape {ds[vnames[0]].shape}")
    return ds

print("\n-- Loading IFS parameters (Greenland subset) --")
ifs_t2m  = load_ifs_param(str(ifs_grib), {"shortName": "2t"},  label="2-m temperature")
ifs_msl  = load_ifs_param(str(ifs_grib), {"shortName": "msl"}, label="MSLP")
ifs_tp   = load_ifs_param(str(ifs_grib), {"shortName": "tp"},  label="Total precip")
ifs_sf   = load_ifs_param(str(ifs_grib), {"shortName": "sf"},  label="Snowfall (total solid precip)") * 1000  # Convert from kg/m2 to m (assuming water density of 1000 kg/m3)

# 10-m wind speed: compute from u10/v10 (10m ws not stored as a separate GRIB field)
ifs_u10  = load_ifs_param(str(ifs_grib), {"shortName": "10u"}, label="10-m U wind")
ifs_v10  = load_ifs_param(str(ifs_grib), {"shortName": "10v"}, label="10-m V wind")
ifs_ws10 = np.sqrt(ifs_u10["u10"] ** 2 + ifs_v10["v10"] ** 2)
ifs_ws10.name = "ws10"
ifs_ws10 = ifs_ws10.to_dataset()
print(f"  OK 10-m wind speed: computed, shape {ifs_ws10['ws10'].shape}")

print("\nDone - 5 IFS fields loaded for Greenland.")

In [ ]:
#Create an Animated contour plot of MSLP + Snowfall over Greenland
msl_da = ifs_msl["msl"]
sf_da  = ifs_sf["sf"]

# Convert MSLP from Pa to hPa
if float(msl_da.max()) > 2000:
    msl_da = msl_da / 100.0

lats = msl_da.latitude.values
lons = msl_da.longitude.values
steps = msl_da.step.values

# Colour-scale range for snowfall
sf_max = float(sf_da.max())
sf_levels = np.linspace(0, max(sf_max, 0.001), 15)

# MSLP contour levels (every 4 hPa)
msl_min = float(msl_da.min())
msl_max_val = float(msl_da.max())
msl_levels = np.arange(np.floor(msl_min / 4) * 4, np.ceil(msl_max_val / 4) * 4 + 1, 4)

proj = ccrs.LambertConformal(central_longitude=-42, central_latitude=72)
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={"projection": proj})

def draw_ifs_frame(idx):
    ax.clear()
    ax.set_extent([-75, -10, 58, 85], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, linestyle="--")
    ax.add_feature(cfeature.LAND, facecolor="0.92", zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor="lightcyan", zorder=0)

    sf_vals  = sf_da.isel(step=idx).values
    msl_vals = msl_da.isel(step=idx).values
    step_h = int(steps[idx] / np.timedelta64(1, "h"))

    # Filled contours: snowfall (total solid precip)
    cf = ax.contourf(
        lons, lats, sf_vals,
        levels=sf_levels,
        cmap="Blues",
        extend="max",
        transform=ccrs.PlateCarree(),
    )
    # Contour lines: MSLP
    cs = ax.contour(
        lons, lats, msl_vals,
        levels=msl_levels,
        colors="k",
        linewidths=0.8,
        transform=ccrs.PlateCarree(),
    )
    ax.clabel(cs, fmt="%.0f", fontsize=7, inline=True)
    ax.set_title(f"IFS MSLP (hPa, contours) + Snowfall (filled)\nT+{step_h:03d}h", fontsize=12)
    return cf, cs

# Draw first frame for the colorbar
cf0, _ = draw_ifs_frame(0)
cbar = fig.colorbar(cf0, ax=ax, orientation="horizontal", pad=0.05, shrink=0.7)
cbar.set_label(f"Snowfall (mm of water equivalent)")

anim = animation.FuncAnimation(fig, draw_ifs_frame, frames=len(steps), interval=400, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

## Similar using DMI Open Data
Below is an example of a similar code snippet as above to download DMI's data for IG. Unfortunately, one cannot download just parameters needed, but must download the full dataset for each timestep. This makes the data load very heavy and thus not very useful for small excercises like the one today.

In [ ]:
# Query DMI Open Data for latest HARMONIE IG SF model run
API_BASE = "https://dmigw.govcloud.dk/v1/forecastdata"
COLLECTION = "harmonie_ig_sf"

# Fetch recent items and find the latest model run
resp = requests.get(
    f"{API_BASE}/collections/{COLLECTION}/items",
    params={"limit": 10},
    timeout=30,
)
resp.raise_for_status()
features = resp.json()["features"]
# Pick the most recent modelRun from the returned items
model_run = max(f["properties"]["modelRun"] for f in features)
print(f"Latest model run: {model_run}")

# Fetch every item that belongs to this model run
all_items, offset = [], 0
while True:
    r = requests.get(
        f"{API_BASE}/collections/{COLLECTION}/items",
        params={"modelRun": model_run, "limit": 100, "offset": offset},
        timeout=30,
    )
    r.raise_for_status()
    feats = r.json()["features"]
    if not feats:
        break
    all_items.extend(feats)
    if len(feats) < 100:
        break
    offset += len(feats)

all_items.sort(key=lambda x: x["properties"]["datetime"])
print(f"Found {len(all_items)} forecast steps for model run {model_run}")

# Download GRIB files
# NOTE: The DMI STAC API serves complete GRIB files per forecast step;
# parameter-level filtering is not available at the download level.
data_dir = Path("grib_data")
data_dir.mkdir(exist_ok=True)

grib_paths = []
for item in all_items[:5]:# Limit to first 5 steps for this example; remove "[:5]" to download all steps
    href = item["asset"]["data"]["href"]
    fname = data_dir / item["id"]
    if not fname.exists():
        print(f"  Downloading {fname.name}")
        dl = requests.get(href, timeout=180)
        dl.raise_for_status()
        fname.write_bytes(dl.content)
    grib_paths.append(str(fname))
print(f"\n{len(grib_paths)} GRIB files ready in {data_dir}/")

print("\n-- Variables in first file --")
sample_dsets = cfgrib.open_datasets(grib_paths[0])
for i, ds in enumerate(sample_dsets):
    print(f"  sub-dataset {i}: {list(ds.data_vars)}")

def load_param(paths, filter_keys, label=""):
    """Open many GRIB files with a cfgrib filter and concatenate in time."""
    pieces = []
    for p in paths:
        try:
            ds = xr.open_dataset(
                p,
                engine="cfgrib",
                backend_kwargs={"filter_by_keys": filter_keys, "indexpath": ""},
            )
            pieces.append(ds)
        except Exception:
            pass
    if not pieces:
        print(f"  WARNING: no data found for filter {filter_keys}")
        return None
    combined = xr.concat(pieces, dim="valid_time").sortby("valid_time")
    vnames = list(combined.data_vars)
    print(f"  OK {label or vnames}: {len(pieces)} steps, shape {combined[vnames[0]].shape}")
    return combined

#Load only 5 parameters
print("\n-- Loading parameters --")
ds_t2m   = load_param(grib_paths, {"shortName": "2t"},     label="2-m temperature")
ds_ws10  = load_param(grib_paths, {"shortName": "10si"},   label="10-m wind speed")
ds_tp    = load_param(grib_paths, {"shortName": "tp"},     label="Total precip")
ds_mslp  = load_param(grib_paths, {"shortName": "pres", "typeOfLevel": "heightAboveSea", "level": 0},
                       label="MSLP (pres heightAboveSea 0)")
ds_solid = load_param(grib_paths, {"shortName": "titspf"}, label="Total solid precip")

print("\nDone - all five fields loaded.")